# Daily Challenge: Classification with Neural Networks in TensorFlow
## Week 6 — Day 1 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build, improve and visualize neural network classifiers on a circles dataset.

**Steps:** Classification types → Dataset → Basic model → Improved model → Decision boundary → Activation functions → Train/Test split → Evaluation → Summary

## Step 1 — Understanding Classification Types

### Binary Classification
**Definition:** The model predicts one of **two possible classes** (0 or 1).  
**Example:** Email spam detection — is this email spam or not spam?  
**Output activation:** Sigmoid → outputs a probability between 0 and 1.  
**Loss function:** Binary Crossentropy.

### Multi-class Classification
**Definition:** The model predicts **one class** from **more than two** possible classes.  
**Example:** Handwritten digit recognition (MNIST) — is this digit 0, 1, 2, … or 9?  
**Output activation:** Softmax → outputs probabilities that sum to 1 across all classes.  
**Loss function:** Categorical Crossentropy (or Sparse Categorical Crossentropy).

### Multi-label Classification
**Definition:** The model can assign **multiple classes simultaneously** to a single sample.  
**Example:** Movie genre tagging — a film can be labeled as both *Action* and *Comedy* at the same time.  
**Output activation:** Sigmoid on each output neuron independently.  
**Loss function:** Binary Crossentropy applied per label.

| Type | Classes | Example | Output Activation |
|---|---|---|---|
| Binary | 2 | Spam / Not Spam | Sigmoid |
| Multi-class | > 2 (mutually exclusive) | Digit 0–9 | Softmax |
| Multi-label | > 2 (can overlap) | Movie genres | Sigmoid per label |

## Step 2 — Setup & Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

print(f"TensorFlow {tf.__version__} ✓")
print(f"NumPy {np.__version__} ✓")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Create the dataset
samples = 1000
X, y = make_circles(samples, noise=0.03, random_state=42)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFirst 5 X values:\n', X[:5])
print('\nFirst 5 y values:', y[:5])
print(f'\nClass distribution: {np.sum(y==0)} class 0, {np.sum(y==1)} class 1')

# Visualize the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], color='#4e91d4', alpha=0.7, s=20, label='Class 0 (outer)')
plt.scatter(X[y==1, 0], X[y==1, 1], color='#e05c5c', alpha=0.7, s=20, label='Class 1 (inner)')
plt.title('make_circles Dataset — 1000 samples', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('dc_plot1_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 3 — Basic Neural Network Model

In [ ]:
# Basic model: single dense layer
model_basic = tf.keras.Sequential([
    tf.keras.layers.Dense(1, activation='sigmoid', input_shape=(2,))
], name='basic_model')

model_basic.compile(
    loss='binary_crossentropy',
    optimizer='sgd',
    metrics=['accuracy']
)

model_basic.summary()

# Train
history_basic = model_basic.fit(X, y, epochs=100, verbose=0)

train_loss, train_acc = model_basic.evaluate(X, y, verbose=0)
print(f"\nBasic model — Loss: {train_loss:.4f} | Accuracy: {train_acc*100:.2f}%")
print("→ Single layer with linear activation cannot learn the circular boundary.")

## Step 4 — Improved Model (More Layers, More Neurons, Adam)

In [ ]:
# Improved model: 2 hidden layers, more neurons, Adam optimizer
model_improved = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')
], name='improved_model')

model_improved.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_improved.summary()

# Train for more epochs
history_improved = model_improved.fit(X, y, epochs=300, verbose=0)

train_loss, train_acc = model_improved.evaluate(X, y, verbose=0)
print(f"\nImproved model — Loss: {train_loss:.4f} | Accuracy: {train_acc*100:.2f}%")

# Training curve
plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(history_basic.history['accuracy'],    label='Basic (SGD)', color='#e05c5c', lw=2)
plt.plot(history_improved.history['accuracy'], label='Improved (Adam)', color='#4e91d4', lw=2)
plt.title('Accuracy — Basic vs Improved', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
plt.plot(history_basic.history['loss'],    label='Basic (SGD)', color='#e05c5c', lw=2)
plt.plot(history_improved.history['loss'], label='Improved (Adam)', color='#4e91d4', lw=2)
plt.title('Loss — Basic vs Improved', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dc_plot2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 5 — Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(model, X, y, title='Decision Boundary', ax=None):
    """Visualize decision boundary for a 2D binary classification model."""
    close = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))

    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid, verbose=0).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.5)
    ax.contour(xx, yy, Z, levels=[0.5], colors='white', linewidths=2)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='#4e91d4', s=15, alpha=0.8, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='#e05c5c', s=15, alpha=0.8, label='Class 1')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.legend(fontsize=9)
    if close:
        plt.tight_layout()
        plt.show()

# Compare basic vs improved decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_decision_boundary(model_basic,    X, y, title='Basic Model (SGD, 1 layer)', ax=axes[0])
plot_decision_boundary(model_improved, X, y, title='Improved Model (Adam, 2 layers)', ax=axes[1])
plt.suptitle('Decision Boundaries Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot3_decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 6 — Activation Functions: ReLU vs Sigmoid

In [ ]:
def build_model(activation, name):
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation=activation, input_shape=(2,)),
        tf.keras.layers.Dense(32, activation=activation),
        tf.keras.layers.Dense(1,  activation='sigmoid')
    ], name=name)
    m.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return m

# Build and train both models on full data for comparison
model_relu    = build_model('relu',    'relu_model')
model_sigmoid = build_model('sigmoid', 'sigmoid_model')

history_relu    = model_relu.fit(X, y,    epochs=200, verbose=0)
history_sigmoid = model_sigmoid.fit(X, y, epochs=200, verbose=0)

_, acc_relu    = model_relu.evaluate(X, y, verbose=0)
_, acc_sigmoid = model_sigmoid.evaluate(X, y, verbose=0)

print(f"ReLU model    accuracy: {acc_relu*100:.2f}%")
print(f"Sigmoid model accuracy: {acc_sigmoid*100:.2f}%")

# Compare decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_decision_boundary(model_relu,    X, y, title=f'ReLU Activation ({acc_relu*100:.1f}%)', ax=axes[0])
plot_decision_boundary(model_sigmoid, X, y, title=f'Sigmoid Activation ({acc_sigmoid*100:.1f}%)', ax=axes[1])
plt.suptitle('ReLU vs Sigmoid — Decision Boundaries', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot4_activations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")
print("\nNote: ReLU generally converges faster and avoids the vanishing gradient problem.")

## Steps 7 & 8 — Train/Test Split + Final Model Evaluation

In [ ]:
# Step 7 — 80/20 Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

# Final model: best configuration (ReLU + Adam + 2 hidden layers)
tf.random.set_seed(42)
model_final = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')
], name='final_model')

model_final.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

# Step 8 — Train and evaluate
history_final = model_final.fit(
    X_train, y_train,
    epochs=200,
    validation_data=(X_test, y_test),
    verbose=0
)

train_loss, train_acc = model_final.evaluate(X_train, y_train, verbose=0)
test_loss,  test_acc  = model_final.evaluate(X_test,  y_test,  verbose=0)

print(f"\n{'='*45}")
print(f"  FINAL MODEL RESULTS")
print(f"{'='*45}")
print(f"  Train — Loss: {train_loss:.4f} | Accuracy: {train_acc*100:.2f}%")
print(f"  Test  — Loss: {test_loss:.4f}  | Accuracy: {test_acc*100:.2f}%")
print(f"{'='*45}")

In [ ]:
# Final evaluation plots
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

# Training curves
axes[0].plot(history_final.history['accuracy'],     label='Train', color='#4e91d4', lw=2)
axes[0].plot(history_final.history['val_accuracy'], label='Test',  color='#e05c5c', lw=2, linestyle='--')
axes[0].set_title('Final Model — Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Decision boundary on training data
plot_decision_boundary(model_final, X_train, y_train,
                       title=f'Train Decision Boundary ({train_acc*100:.1f}%)', ax=axes[1])

# Decision boundary on test data
plot_decision_boundary(model_final, X_test, y_test,
                       title=f'Test Decision Boundary ({test_acc*100:.1f}%)', ax=axes[2])

plt.suptitle('Final Model — Performance Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dc_plot5_final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Step 9 — Summary & Key Takeaways

**What I learned:**

1. **Classification types matter** — Binary (2 classes), Multi-class (1 of N), and Multi-label (multiple of N) each require different output activations and loss functions. For the circles dataset we used Binary Crossentropy + Sigmoid.

2. **A single dense layer is not enough** — The basic model (1 layer, SGD) failed to learn the circular decision boundary because it can only produce a linear separator. The circles dataset requires a non-linear boundary.

3. **More layers + ReLU = better results** — Adding hidden layers with ReLU activation allows the network to learn non-linear functions. ReLU is preferred over Sigmoid in hidden layers because it doesn't suffer from the vanishing gradient problem.

4. **Adam outperforms SGD** — Adam adapts the learning rate per parameter and converges much faster and more reliably than vanilla SGD.

5. **Visualizing decision boundaries is crucial** — Plotting the decision boundary in 2D reveals exactly what the model has learned. It makes overfitting and underfitting immediately visible.

6. **Hyperparameter tuning matters** — Increasing neurons (64 → 64 → 32) and training epochs significantly improved accuracy from ~50% (basic) to ~99% (final model).

**Importance of visualization:** Data visualization helped identify that the problem is non-linear before building the first model. Without the scatter plot, we might have wasted time with linear models. Decision boundary plots also confirm that the model generalizes well (train and test boundaries look similar).

**Importance of hyperparameter tuning:** The jump from 50% to 99% accuracy came from choosing the right architecture (depth), activation function (ReLU), optimizer (Adam), and training duration. Tuning these is as important as the data itself.